# **Import des modules nécessaires**

In [2]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic
from sklearn.cluster import KMeans #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [3]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",214
1,1865 La confession de Claude.,1865,1,2,"La mansarde entière me réclame les rires, les ...",187
2,1865 La confession de Claude.,1865,1,3,Le grillon chantait; le souffle harmonieux des...,146
3,1865 La confession de Claude.,1865,1,4,"brunes et rieuses filles, étaient reines des m...",190
4,1865 La confession de Claude.,1865,1,5,"Pars cependant, puisque tu as soif de la vie. ...",126


In [4]:
df.shape

(21260, 6)

# **Traitement du Corpus de Zola**

In [5]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

In [6]:
# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=50): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver matin frais manteau brouillard saison so...
1,mansarde entier rire richesse sœur foyer feu j...
2,grillon souffle harmonieux causerie lèvre cœur...
3,brun rieur moisson vendange épi grappe sentier...
4,soif projet soi ferme loyal action rêve vis gr...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [7]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/665 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

In [28]:
hdbscan_model = HDBSCAN( min_cluster_size=25, min_samples=3, metric='euclidean', cluster_selection_method='eom',prediction_data=True)

umap_model = UMAP( n_neighbors=20,n_components=3, min_dist=0.0, metric="cosine",random_state=42)

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [ ]:
hdbscan_model = HDBSCAN( min_cluster_size=25, 
                        min_samples=3, 
                        metric='euclidean', 
                        cluster_selection_method='eom',
                        prediction_data=True)

umap_model = UMAP( n_neighbors=20,
                  n_components=3, 
                  min_dist=0.0, 
                  metric="cosine",
                  random_state=42)


vectorizer_model = CountVectorizer(
    min_df=4,    # Le mot doit apparaître dans au moins 2 segments pour être pris en compte (élimine les fautes ou mots uniques)
    max_df=0.7)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True,
    nr_topics="auto"
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

2026-07-08 14:11:09,837 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-08 14:11:19,547 - BERTopic - Dimensionality - Completed ✓
2026-07-08 14:11:19,548 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-08 14:11:19,819 - BERTopic - Cluster - Completed ✓
2026-07-08 14:11:19,820 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-07-08 14:11:20,051 - BERTopic - Representation - Completed ✓
2026-07-08 14:11:20,052 - BERTopic - Topic reduction - Reducing number of topics
2026-07-08 14:11:20,069 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-08 14:11:20,268 - BERTopic - Representation - Completed ✓
2026-07-08 14:11:20,270 - BERTopic - Topic reduction - Reduced number of topics from 115 to 55


In [30]:
new_topics = topic_model.reduce_outliers(
    df["phrases_lemm"].tolist(), 
    topics, 
    strategy="embeddings",
    embeddings=embeddings
)

# Met à jour le modèle avec ces nouveaux thèmes plus propres
topic_model.update_topics(df["phrases_lemm"].tolist(), topics=new_topics)

2026-07-08 14:11:49,936 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


## **3) Topics Présent**

In [31]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,5024,0_franc_dame_maison_soir,"[franc, dame, maison, soir, mari, porte, fois,...",[face carré nez bec aigle bouche large ferme d...
1,1,1563,1_soleil_noir_haut_blanc,"[soleil, noir, haut, blanc, ciel, ombre, long,...",[pente coteau boisé ville dôme colossal masse ...
2,2,1434,2_amour_cœur_pensée_mort,"[amour, cœur, pensée, mort, tendresse, amant, ...",[coin ombre poltron hagard nouveau individu pa...
3,3,577,3_cœur_larme_heureux_pauvre,"[cœur, larme, heureux, pauvre, bonheur, raison...",[guérie reprise long hérédité perversion démen...
4,4,441,4_porte_lit_chambre_fenêtre,"[porte, lit, chambre, fenêtre, jambe, bruit, p...",[horloge grinça arracher abbé frisson fraîcheu...
5,5,582,5_pauvre_cœur_larme_vou,"[pauvre, cœur, larme, vou, ami, parole, cher, ...",[but secret occasion cher cousin quai désert s...
6,6,269,6_peuple_science_religion_siècle,"[peuple, science, religion, siècle, vérité, ég...",[monument orgueil domination science nom idéal...
7,7,620,7_maman_rire_idée_vrai,"[maman, rire, idée, vrai, bête, chéri, argent,...",[drôle écoute dessert envie mauvais histoire c...
8,8,393,8_abbé_prêtre_curé_affaire,"[abbé, prêtre, curé, affaire, ami, cher, vicai...",[doute matin voyage cardinal secrétaire cardin...
9,9,397,9_franc_fortune_million_argent,"[franc, fortune, million, argent, affaire, mai...",[feuille financier traité année colonne numéro...


In [33]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=20 
)

topic_model.visualize_topics_over_time(topics_over_time) #topics=themes_interet)

19it [00:02,  7.68it/s]


In [14]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('amour', np.float64(0.010121056584973942)), ('cœur', np.float64(0.009084831756686123)), ('pensée', np.float64(0.007537823991038832)), ('amant', np.float64(0.007492269292488328)), ('passion', np.float64(0.007370059831349973)), ('chair', np.float64(0.007060846422774702)), ('tendresse', np.float64(0.006947396262719577)), ('désir', np.float64(0.006406614480843874)), ('mort', np.float64(0.0061420202419122825)), ('existence', np.float64(0.005828656694471155))]

TOPIC 1
[('affaire', np.float64(0.007251659850241547)), ('franc', np.float64(0.006924773237352363)), ('fortune', np.float64(0.006550032608275318)), ('instituteur', np.float64(0.0065095866411592354)), ('fils', np.float64(0.006348002448272873)), ('frère', np.float64(0.00585789404700347)), ('année', np.float64(0.005491552267253789)), ('école', np.float64(0.005399147303825323)), ('ancien', np.float64(0.004995309039621288)), ('politique', np.float64(0.00475055834746283))]

TOPIC 2
[('cher', np.float64(0.009487203800158942)), ('d